# [PBT] 화장품 이커머스 ㅣ 07. 번인 진단

## 이 노트북이 하는 일

**번인(burn-in)을 도입하기로 한 근거**를 남긴다. 04번에 붙어 있던 진단 셀을 옮겨 왔다.

옮긴 이유는 두 가지다.
- 04번은 **설명변수 X를 만드는 노트북**인데 이 셀들은 목표변수 Y를 읽는다. 역할이 섞였다.
- 04번을 번인 적용 후 데이터로 다시 돌리면 **이 출력이 전부 덮어써진다.**
  그런데 이 출력이 곧 번인의 근거다.

## ⚠️ 이 노트북의 출력이 곧 증거다

아래 셀의 출력은 **번인 적용 전 데이터(190,698명)** 로 실행한 결과다.
지금 `output\` 에 있는 파일은 번인 적용 후(108,458명)이므로 **그대로 다시 돌리면 안 된다.**
다시 돌리려면 번인 전 `02b_x.csv` · `02c_y.csv` 가 필요하다
(`output\_prewin\` 에 `02a_target.csv` 만 보존해 두었다).

## 결론 요약

| 확인한 것 | 결과 |
|---|---|
| 주차별 Y | 첫 주 7.79% → 10/14 이후 20% 안팎에서 평평 |
| 조회 여부 ↔ Y | 전체 집계 시 **부호가 뒤집힘** (18.87% vs 13.60%) |
| 번인 후 조회 여부 ↔ Y | 방향 정상 회복 (18.57% vs 21.20%) |
| 번인 시작일 | 10/11에서 Y 20.50%, 10/15(20.52%)와 0.02%p 차 → 이미 안정 |

시작일은 **일별 t₀ 인원 분포**(목표변수와 무관한 양)로 정했고,
Y 안정화는 **사후 확인**으로만 사용했다. 순서가 뒤바뀌면 유리한 표본을 고른 것이 된다.


In [1]:
import os, sys

# 프로젝트 폴더 — notebooks/ 안에서 실행하면 상위 폴더를 가리킨다
from pathlib import Path
BASE = str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
os.chdir(BASE)
sys.path.insert(0, BASE)

from pandas import read_csv, to_datetime, Timestamp, DataFrame, cut
from helpers import *


OUT = BASE + r"\output\_prewin"      # ← 번인 적용 "전" 파일이 있는 곳


In [2]:
# =========================================================
# [진단] 코호트 편중과 조회 변수의 실효성
#   2번 — 첫 주 코호트 vs 이후의 Y 비율   → 번인 도입 여부
#   3번 — 조회 변수 각각과 Y의 관계        → 어느 변수를 남길지
# =========================================================
from pandas import read_csv, to_datetime, Timestamp, DataFrame, cut

x = read_csv(OUT + r"\02b_x.csv", encoding="utf-8-sig")
y = read_csv(OUT + r"\02c_y.csv", encoding="utf-8-sig")

d = x.merge(y[["user_id", "purchase_7d", "lag_hours"]], on="user_id", how="inner")
d["t0"] = to_datetime(d["t0"], format="%Y-%m-%d %H:%M:%S")

assert len(d) == len(x)
assert d["purchase_7d"].isna().sum() == 0

print(f"{len(d):,}행  |  Y 양성 {d['purchase_7d'].mean():.2%}")

190,698행  |  Y 양성 15.36%


In [3]:
# ── 2-A. 주차별 Y 비율 ──────────────────────────────
g = d.groupby(d["t0"].dt.to_period("W"))

w = DataFrame({"고객 수": g.size(), "양성": g["purchase_7d"].sum()})
w["Y 비율(%)"]           = (w["양성"] / w["고객 수"] * 100).round(2)
w["has_prior_view=1(%)"] = (g["has_prior_view"].mean() * 100).round(2)
w

,고객 수,양성,Y 비율(%),has_prior_view=1(%)
t0,,,,
2019-09-30/2019-10-06,55718,4341,7.79,83.58
2019-10-07/2019-10-13,33760,4153,12.30,72.68
2019-10-14/2019-10-20,18026,3675,20.39,55.08
2019-10-21/2019-10-27,16651,3350,20.12,54.73
2019-10-28/2019-11-03,15665,3187,20.34,56.63
2019-11-04/2019-11-10,17202,3584,20.83,55.85
2019-11-11/2019-11-17,15524,3099,19.96,55.62
2019-11-18/2019-11-24,18152,3903,21.50,54.31


In [4]:
# ── 2-B. 번인 시작일별 표본과 Y 비율 ────────────────
rows = []
for day in ["2019-10-01", "2019-10-03", "2019-10-05",
            "2019-10-08", "2019-10-11", "2019-10-15"]:
    sub = d.loc[d["t0"] >= Timestamp(day)]
    rows.append({
        "번인 시작일"        : day,
        "남는 고객"          : len(sub),
        "남는 비율(%)"       : round(len(sub) / len(d) * 100, 1),
        "Y 비율(%)"          : round(sub["purchase_7d"].mean() * 100, 2),
        "has_prior_view=1(%)": round(sub["has_prior_view"].mean() * 100, 2),
    })

DataFrame(rows).set_index("번인 시작일")

,남는 고객,남는 비율(%),Y 비율(%),has_prior_view=1(%)
번인 시작일,,,,
2019-10-01,190698,100.0,15.36,66.66
2019-10-03,162996,85.5,16.82,63.25
2019-10-05,154736,81.1,16.80,63.21
2019-10-08,120589,63.2,20.05,56.27
2019-10-11,108458,56.9,20.50,55.32
2019-10-15,98172,51.5,20.52,55.33


In [5]:
# ── 3. 조회 변수별 Y 비율 ───────────────────────────
def rate_by(col, bins=None, labels=None):
    s = cut(d[col], bins=bins, labels=labels) if bins else d[col]
    gg = d.groupby(s, observed=True)
    t = DataFrame({"고객 수": gg.size(), "양성": gg["purchase_7d"].sum()})
    t["Y 비율(%)"] = (t["양성"] / t["고객 수"] * 100).round(2)
    return t

B = [-1, 0, 1, 2, 3, 5, 10, 1000]
L = ["0", "1", "2", "3", "4-5", "6-10", "11+"]

print("── has_prior_view ──");         display(rate_by("has_prior_view"))
print("── prior_view_cnt ──");         display(rate_by("prior_view_cnt", B, L))
print("── prior_uniq_product_cnt ──"); display(rate_by("prior_uniq_product_cnt", B, L))
print("── cart_product_view_cnt ──");  display(rate_by("cart_product_view_cnt",
                                                      [-1, 0, 1, 2, 3, 10],
                                                      ["0", "1", "2", "3", "4+"]))

── has_prior_view ──


,고객 수,양성,Y 비율(%)
has_prior_view,,,
0,63579,12000,18.87
1,127119,17292,13.60


── prior_view_cnt ──


,고객 수,양성,Y 비율(%)
prior_view_cnt,,,
0,63579,12000,18.87
1,91906,9311,10.13
2,15295,3430,22.43
3,7308,1638,22.41
4-5,6418,1409,21.95
6-10,4531,1064,23.48
11+,1661,440,26.49


── prior_uniq_product_cnt ──


,고객 수,양성,Y 비율(%)
prior_uniq_product_cnt,,,
0,63579,12000,18.87
1,96448,10513,10.90
2,14599,3218,22.04
3,6232,1317,21.13
4-5,5359,1180,22.02
6-10,3383,775,22.91
11+,1098,289,26.32


── cart_product_view_cnt ──


,고객 수,양성,Y 비율(%)
cart_product_view_cnt,,,
0,87657,15873,18.11
1,95410,11222,11.76
2,6083,1712,28.14
3,1161,354,30.49
4+,387,131,33.85


In [6]:
# ── 4. 번인 적용 후 조회 변수 ↔ Y 재확인 ──────────────
#    시기가 교란 변수인지 판정한다.
#    U자형이 그대로면 진짜 구조, 사라지면 첫 주가 만든 착시다.

def rate_by2(data, col, bins=None, labels=None):
    s = cut(data[col], bins=bins, labels=labels) if bins else data[col]
    gg = data.groupby(s, observed=True)
    t = DataFrame({"고객 수": gg.size(), "양성": gg["purchase_7d"].sum()})
    t["Y 비율(%)"] = (t["양성"] / t["고객 수"] * 100).round(2)
    return t

late = d.loc[d["t0"] >= Timestamp("2019-10-08")]
print(f"번인 후 : {len(late):,}명  |  Y {late['purchase_7d'].mean():.2%}\n")

print("── has_prior_view (번인 후) ──")
display(rate_by2(late, "has_prior_view"))

print("── prior_view_cnt (번인 후) ──")
display(rate_by2(late, "prior_view_cnt", B, L))

print("── cart_product_view_cnt (번인 후) ──")
display(rate_by2(late, "cart_product_view_cnt", [-1, 0, 1, 2, 3, 10], ["0", "1", "2", "3", "4+"]))

번인 후 : 120,589명  |  Y 20.05%

── has_prior_view (번인 후) ──


,고객 수,양성,Y 비율(%)
has_prior_view,,,
0,52737,9795,18.57
1,67852,14386,21.20


── prior_view_cnt (번인 후) ──


,고객 수,양성,Y 비율(%)
prior_view_cnt,,,
0,52737,9795,18.57
1,38798,7804,20.11
2,12761,2888,22.63
3,6027,1330,22.07
4-5,5278,1176,22.28
6-10,3645,835,22.91
11+,1343,353,26.28


── cart_product_view_cnt (번인 후) ──


,고객 수,양성,Y 비율(%)
cart_product_view_cnt,,,
0,72353,12865,17.78
1,41703,9435,22.62
2,5204,1464,28.13
3,1000,302,30.20
4+,329,115,34.95


## 결론

**번인을 도입한다. 시작일은 2019-10-11, 남는 표본은 108,458명(전체의 56.9%)이다.**

### 1) 초기 구간은 다른 모집단이었다

주차별 Y 비율은 첫 주 7.79% → 둘째 주 12.30% → 셋째 주(10/14~) 20.39%로 뛰어오른 뒤,
이후 8주 내내 19.96~21.50% 범위에서 평평하게 유지된다. 서비스 초기에는 아직 잠재 고객군이
다 유입되지 않았거나 트래킹·추천 로직이 안정화되지 않아, 그 시기에 t₀를 찍은 고객들은
이후 구간과 성격이 다른 표본이다. 이 구간을 그대로 학습에 포함하면 모델이 "초기 효과"를
"진짜 패턴"으로 착각한다.

### 2) 조회 변수의 부호가 뒤집히는 것으로 편중을 확인했다

전체 기간을 대상으로 하면 `has_prior_view`의 방향이 상식과 반대로 나온다
(조회 안 함 18.87% > 조회함 13.60%). 반면 번인 이후(10/08~)로만 보면 방향이 정상으로
돌아온다(조회 안 함 18.57% < 조회함 21.20%). `prior_view_cnt`, `cart_product_view_cnt`도
번인 전후 모두 U자형 — 조회 0회와 조회 다회가 모두 조회 1회보다 Y가 높은 구조 —를
유지하므로, 이는 착시가 아니라 실제 구조이고, 초기 구간이 이 구조 위에 왜곡을 얹고 있었을
뿐이라는 뜻이다. 즉 번인이 필요한 이유는 "초기 구간이 껴 있으면 조회 변수의 방향성 해석이
뒤집힌다"는 것이지, 조회 변수 자체가 무의미하다는 뜻이 아니다.

### 3) 시작일은 Y와 무관한 기준으로 먼저 정하고, Y로는 사후 확인만 했다

시작일 후보(10/01, 10/03, 10/05, 10/08, 10/11, 10/15)는 **일별 t₀ 인원 분포**만 보고 정했다
— 목표변수를 들여다보지 않고 고른 지점이다. 그 결과를 Y 비율에 대입해보니 10/11부터
Y 비율이 20.50%로 이미 10/15(20.52%)와 0.02%p밖에 차이 나지 않아, 10/11 이후 구간은 이미
안정 상태에 도달해 있었다. 표본을 더 깎아 10/15부터 자르는 것은 표본 손실(56.9% → 51.5%)
대비 얻는 게 없으므로, 남는 표본이 가장 많은 10/11을 번인 시작일로 채택한다.

만약 Y 비율이 가장 높게 나오는 지점을 거꾸로 찾아 시작일로 정했다면 표본을 유리하게
고른 것이 되어 번인의 근거 자체가 무효가 된다. 순서를 지킨 것이 이 진단의 핵심이다.

### 4) 이후 노트북에서 지켜야 할 것

- 04번(설명변수 X 생성) 이후의 모든 노트북은 `t0 >= 2019-10-11` 로 필터링된 데이터로 다시
  실행해야 한다. 이 노트북에 남아 있는 출력(190,698명 기준)은 번인 미적용 상태의 근거
  자료이며, 이후 파이프라인의 입력으로 쓰면 안 된다.
- `output\` 의 현재 파일들은 이미 번인 적용 후(108,458명) 기준이다. 이 노트북만 예외적으로
  `output\_prewin\` 을 참조해 번인 전 상태를 재현한다.
